# Partie 3 — Feature Engineering & Modélisation ML
**Entrée** : `immobilier_clean.csv`  
**Sortie** : `immobilier_predictions.csv`

## 1. Installation & Imports

In [ ]:
pip install pyspark

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','pyspark','--quiet'])
print('OK')

In [ ]:
import os, shutil, re, subprocess, time
import pandas as pd

subprocess.run('pkill -f SparkSubmit',   shell=True, capture_output=True)
subprocess.run('pkill -f pyspark',       shell=True, capture_output=True)
subprocess.run('pkill -f GatewayServer', shell=True, capture_output=True)
time.sleep(2)

try:
    from pyspark import SparkContext
    if SparkContext._active_spark_context:
        SparkContext._active_spark_context.stop()
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
except Exception:
    pass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName('Immobilier') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.driver.maxResultSize', '1g') \
    .config('spark.memory.offHeap.enabled', 'true') \
    .config('spark.memory.offHeap.size', '512m') \
    .master('local[2]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
DATA_DIR = os.getcwd()
print(f'Spark {spark.version} — OK')

## 2. Chargement

In [ ]:
df = spark.read.csv(
    os.path.join(DATA_DIR, 'immobilier_clean.csv'),
    header=True, inferSchema=True
)
print(f'Lignes chargees : {df.count()}')
df.printSchema()

## 3. Calcul du prix au m²

`prix_m2 = prix / surface` — normalise les biens par leur surface pour comparer des marchés hétérogènes.

In [ ]:
df = df.withColumn('prix_m2', F.col('prix') / F.col('surface'))
df = df.filter(
    F.col('prix_m2').isNotNull() &
    (F.col('prix_m2') > 200)    &
    (F.col('prix_m2') < 30000)  &
    (F.col('surface') >= 9)
)
print(f'Lignes apres filtrage prix_m2 : {df.count()}')

df.groupBy('type_bien') \
  .agg(F.count('*').alias('nb'),
       F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen'),
       F.round(F.min('prix_m2'),  0).alias('min'),
       F.round(F.max('prix_m2'),  0).alias('max')) \
  .orderBy(F.desc('prix_m2_moyen')).show()

df.groupBy('type_transaction') \
  .agg(F.count('*').alias('nb'),
       F.round(F.mean('prix_m2'), 0).alias('prix_m2_moyen')).show()

## 4. Feature Engineering

- Extraction du département (2 premiers chiffres du CP)
- Remplacement des chaînes vides par `inconnu`
- StringIndexer + OneHotEncoder + VectorAssembler

In [ ]:
df = df.withColumn('nb_pieces',   F.coalesce(F.col('nb_pieces'),   F.lit(3.0))) \
       .withColumn('nb_chambres', F.coalesce(F.col('nb_chambres'), F.lit(1.0)))

df = df.withColumn(
    'departement',
    F.when(F.col('code_postal').isNotNull(),
           F.substring(F.col('code_postal').cast('string'), 1, 2)
    ).otherwise('00')
)

# Remplacement chaines vides — StringIndexer refuse les ""
cat_cols = ['type_bien', 'type_transaction', 'source', 'departement']
for c in cat_cols:
    df = df.withColumn(c,
        F.when((F.col(c).isNull()) | (F.trim(F.col(c)) == ''), F.lit('inconnu'))
         .otherwise(F.col(c)))

num_cols  = ['surface', 'nb_pieces', 'nb_chambres']
indexers  = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat_cols]
encoders  = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in cat_cols]
assembler = VectorAssembler(
    inputCols=num_cols + [c+'_ohe' for c in cat_cols],
    outputCol='features', handleInvalid='skip'
)
print('Features :', num_cols + cat_cols)
print('Target   : prix_m2')

## 5. Split Train / Test (80% / 20%)

In [ ]:
df_ml = df.select(
    *num_cols, *cat_cols,
    'prix_m2', 'prix', 'ville', 'code_postal'
).dropna(subset=['prix_m2'])

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# Cache obligatoire — evite de recalculer tout le pipeline a chaque modele
train_df.cache()
test_df.cache()
train_df.count()
test_df.count()

print(f'Train : {train_df.count()} lignes (cache)')
print(f'Test  : {test_df.count()} lignes (cache)')

## 6. Entraînement — Régression Linéaire, Random Forest, GBT

In [ ]:
evaluator      = RegressionEvaluator(labelCol='prix_m2', predictionCol='prediction', metricName='r2')
evaluator_rmse = RegressionEvaluator(labelCol='prix_m2', predictionCol='prediction', metricName='rmse')

results = []

# ── Regression Lineaire ──────────────────────────────────────────
lr = LinearRegression(labelCol='prix_m2', featuresCol='features', maxIter=50)
model_lr = Pipeline(stages=indexers + encoders + [assembler, lr]).fit(train_df)
preds_lr = model_lr.transform(test_df)
r2_lr   = evaluator.evaluate(preds_lr)
rmse_lr = evaluator_rmse.evaluate(preds_lr)
preds_lr.unpersist()
print(f'Regression Lineaire — R2={r2_lr:.4f} | RMSE={rmse_lr:.2f} euros/m2')
results.append(('Regression Lineaire', r2_lr, rmse_lr, model_lr))

# ── Random Forest ────────────────────────────────────────────────
rf = RandomForestRegressor(labelCol='prix_m2', featuresCol='features',
                           numTrees=30, maxDepth=4, seed=42)
model_rf = Pipeline(stages=indexers + encoders + [assembler, rf]).fit(train_df)
preds_rf = model_rf.transform(test_df)
r2_rf   = evaluator.evaluate(preds_rf)
rmse_rf = evaluator_rmse.evaluate(preds_rf)
preds_rf.unpersist()
print(f'Random Forest       — R2={r2_rf:.4f} | RMSE={rmse_rf:.2f} euros/m2')
results.append(('Random Forest', r2_rf, rmse_rf, model_rf))

# ── GBT ──────────────────────────────────────────────────────────
gbt = GBTRegressor(labelCol='prix_m2', featuresCol='features',
                   maxIter=15, maxDepth=3, seed=42)
model_gbt = Pipeline(stages=indexers + encoders + [assembler, gbt]).fit(train_df)
preds_gbt = model_gbt.transform(test_df)
r2_gbt   = evaluator.evaluate(preds_gbt)
rmse_gbt = evaluator_rmse.evaluate(preds_gbt)
preds_gbt.unpersist()
print(f'GBT                 — R2={r2_gbt:.4f} | RMSE={rmse_gbt:.2f} euros/m2')
results.append(('GBT', r2_gbt, rmse_gbt, model_gbt))

## 7. Comparaison des modèles

In [ ]:
print(f'{"Modele":<25} {"R2":>8} {"RMSE":>12}')
print('-' * 48)
best_r2 = max(r[1] for r in results)
for name, r2, rmse, _ in sorted(results, key=lambda x: -x[1]):
    flag = ' <-- MEILLEUR' if r2 == best_r2 else ''
    print(f'{name:<25} {r2:>8.4f} {rmse:>12.2f}{flag}')

best_name, best_r2, best_rmse, best_model = max(results, key=lambda x: x[1])
print(f'\nModele selectionne : {best_name}')

## 8. Export des prédictions

In [ ]:
df_preds = best_model.transform(df_ml) \
    .withColumn('prix_m2_predit', F.round(F.col('prediction'), 2)) \
    .select('ville', 'code_postal', *cat_cols,
            F.round('surface', 1).alias('surface'),
            F.round('prix', 0).alias('prix'),
            'nb_pieces', 'nb_chambres',
            F.round('prix_m2', 2).alias('prix_m2'),
            'prix_m2_predit')

out = os.path.join(DATA_DIR, 'immobilier_predictions.csv')
if os.path.exists(out):
    shutil.rmtree(out) if os.path.isdir(out) else os.remove(out)
df_preds.toPandas().to_csv(out, index=False, encoding='utf-8')
print(f'immobilier_predictions.csv exporte : {df_preds.count()} lignes')

In [ ]:
spark.stop()
print('Spark termine.')